# 10 · ASR baseline (T1_Speech)

Whisper large-v3-turbo on the **dev** audio → `asr/dev_en_whisper-large-v3-turbo.jsonl`
(one row per clip: `id, transcript, wer`). WER is computed against the text `question` from `qa_*` with the same `id`.

Backend is picked by `default_backend()`: `mlx` on Apple Silicon, `hf` on a CUDA GPU (Kaggle),
`ct2` (faster-whisper, int8 CPU) otherwise, e.g. on an Intel Mac. All three decode greedily, without
temperature fallback. The `hf` backend reads audio files with `ffmpeg` (preinstalled on Kaggle and Colab).
Re-running skips clips that are already transcribed.

On Kaggle: GPU T4, Internet On, secret `HF_TOKEN` attached. The first cell clones the repo and downloads the data; it is skipped locally.


In [1]:
import os
if os.path.exists("/kaggle"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["MMCQA_OUT"] = "/kaggle/working"
    !git clone -q https://github.com/mohinska/mmculture-qa.git /kaggle/working/mmculture-qa
    %cd /kaggle/working/mmculture-qa
    !pip install -q -r requirements.txt
    !python shared/download_data.py dev --task sqa

/kaggle/working/mmculture-qa
download archives/audio_mena_en_dev.zip

archives/audio_mena_en_dev.zip: downloading bytes: ████████▉|  225MB, 19.5MB/s  
archives/audio_mena_en_dev.zip: downloading bytes: █████████|  225MB, 20.5MB/s  
archives/audio_mena_en_dev.zip: reconstructing file: 100%|█|  225MB /  225MB, 21
  unzip -> /kaggle/working/mmculture-qa/data/archives
download archives/audio_mena_en_devtest.zip

archives/audio_mena_en_devtest.zip: downloading bytes: ████▋|  217MB, 17.4MB/s  
archives/audio_mena_en_devtest.zip: downloading bytes: █████|  228MB, 19.9MB/s  
archives/audio_mena_en_devtest.zip: reconstructing file: 100%|█|  228MB /  228MB
  unzip -> /kaggle/working/mmculture-qa/data/archives
download archives/images_mena_dev.zip

archives/images_mena_dev.zip: downloading bytes: ███████▍   |  123MB, 8.69MB/s  
archives/images_mena_dev.zip: downloading bytes: ██████████▋|  177MB, 14.4MB/s  
archives/images_mena_dev.zip: downloading bytes: ███████████|  182MB, 17.2MB/s  
archives/

In [2]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").is_dir())
sys.path[:0] = [str(ROOT / "shared"), str(ROOT / "t1_speech")]

from data import check_ids, check_media, load_jsonl, load_split
from metrics import wer_one
from speech import (WER_BINS, asr_summary, default_backend, missed_names, run_asr,
                    transcribe, wer_bin, wer_refs)

print("ASR backend:", default_backend())

ASR backend: hf


## 1. Data check
Expected: 1000 rows, 0 duplicate ids, 0 missing audio/images.

In [3]:
rows = load_split("sqa", "dev")
print(len(rows), check_ids(rows))
print(check_media(rows))
print(rows[0])

1000 {'rows': 1000, 'unique_ids': 1000, 'duplicates': 0}
{'image_path': {'total': 1000, 'missing': 0, 'examples': []}, 'audio_path': {'total': 1000, 'missing': 0, 'examples': []}}
{'id': '065bd70cd4c6c83fbe80f8138b28ed5500f4e9af6c70465d38dec784d7fda39a', 'image': 'images/065bd70cd4c6c83fbe80f8138b28ed5500f4e9af6c70465d38dec784d7fda39a.jpg', 'audio': 'audio/en/065bd70cd4c6c83fbe80f8138b28ed5500f4e9af6c70465d38dec784d7fda39a.wav', 'answer': 'Lanterns like these often symbolize hospitality and are commonly used in traditional marketplaces for decoration and lighting, creating a warm and inviting atmosphere. They are particularly prevalent in Middle Eastern and North African cultures.', 'row_idx': 0, 'lang': 'en', 'split': 'dev', 'image_path': '/kaggle/working/mmculture-qa/data/archives/images/065bd70cd4c6c83fbe80f8138b28ed5500f4e9af6c70465d38dec784d7fda39a.jpg', 'audio_path': '/kaggle/working/mmculture-qa/data/archives/audio/en/065bd70cd4c6c83fbe80f8138b28ed5500f4e9af6c70465d38dec784d7fda

## 2. Smoke test on 5 clips
Compare by eye with the text question before the full run.

In [4]:
sample = rows[:5]
refs = wer_refs("dev")
hyps = transcribe([r["audio_path"] for r in sample])
for r, h in zip(sample, hyps):
    print(f"WER {wer_one(h, refs[r['id']]):.2f}")
    print("  ref:", refs[r["id"]])
    print("  hyp:", h)

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


WER 0.00
  ref: What cultural significance do lanterns like the ones shown in the image have in traditional marketplaces?
  hyp: What cultural significance do lanterns like the ones shown in the image have in traditional marketplaces?
WER 0.00
  ref: What iconic landmark is visible through the window in this room?
  hyp: What iconic landmark is visible through the window in this room?
WER 0.20
  ref: What is depicted in the photograph labeled 'De Anti-Libanon'?
  hyp: What is depicted in the photograph labeled Dianté Libanon?
WER 0.00
  ref: What is the significance of the ornate crowns and ceremonial banners in this procession?
  hyp: What is the significance of the ornate crowns and ceremonial banners in this procession?
WER 0.00
  ref: What ancient landmark is depicted in the image, and what is its historical significance?
  hyp: What ancient landmark is depicted in the image, and what is its historical significance?


## 3. Full dev run

In [5]:
ASR_MODEL = "whisper-large-v3-turbo"
path = run_asr("dev", model=ASR_MODEL)
path

dev_en_whisper-large-v3-turbo.jsonl: 1000 rows, 0 done, 1000 to go


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  1000/1000  0.40s per clip


PosixPath('/kaggle/working/asr/dev_en_whisper-large-v3-turbo.jsonl')

In [6]:
summary = asr_summary(path, "dev")
summary

{'asr': 'dev_en_whisper-large-v3-turbo.jsonl',
 'split': 'dev',
 'n': 1000,
 'wer_corpus': 0.0069,
 'wer_mean': 0.0072,
 'share_wer0': 0.923,
 'share_wer_gt_0.25': 0.002}

## 4. Where does ASR fail?

In [7]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

asr = pd.DataFrame(load_jsonl(path))
asr["question"] = asr["id"].map(refs)
asr["wer_bin"] = asr["wer"].map(wer_bin)
print(asr["wer_bin"].value_counts().reindex([b[2] for b in WER_BINS]).fillna(0).astype(int))
asr.sort_values("wer", ascending=False)[["wer", "question", "transcript"]].head(15)

wer_bin
0             923
(0, .10]       56
(.10, .25]     19
(.25, .50]      2
> .50           0
Name: count, dtype: int64


,wer,question,transcript
870,0.4167,What does the sign indicate about Camp Men-O-Lan’s history?,What does the sign indicate about Count Menelan's history?
363,0.2727,What activities are the people in this indoor setting engaged in?,What have these other people in this indoor setting engaged in?
412,0.2000,What aspects of the restaurant 'IBN ALSHAM' suggest it is a popular evening dining destination?,"What aspects of the restaurant, Ibn al-Sham, suggested is a popular evening dining destination?"
2,0.2000,What is depicted in the photograph labeled 'De Anti-Libanon'?,What is depicted in the photograph labeled Dianté Libanon?
152,0.1818,What national symbol is displayed on the mousepad in the image?,What national symbol is displayed on the mouse pad in the image?
914,0.1765,"What is Arak, as shown on the label of the blue bottle, and how is it made?","What is Eric? Has shown the label of the blue bottle, and how is it made?"
228,0.1667,What is the vendor using to extract juice from the sugarcane stalks?,What is the vendor using to extract juice from the sugar cane stalks?
900,0.1667,What is the historical significance of the waterwheel shown in the image?,What is the historical significance of the water reels shown in the image?
404,0.1538,What does the microphone labeled 'فلسطين' (Palestine) signify in the context of this image?,What does the microphone labeled Philistine Palestine signify in the context of this image?
244,0.1538,What type of boats are docked along the waterfront in this nighttime scene?,What type of boats are docked along the waterfront in this night time scene?


Capitalized words (names, places, dishes) lost by ASR: the cultural part of the question.

In [8]:
asr["missed"] = [missed_names(q, t) for q, t in zip(asr["question"], asr["transcript"])]
has = asr["missed"].map(len) > 0
print(f"questions with a lost name: {has.mean():.1%}")
pd.Series([w for ws in asr["missed"] for w in ws]).value_counts().head(25)

questions with a lost name: 1.5%


De              1
Anti-Libanon    1
Alahrar         1
Fakhouri        1
Souq            1
Waqif           1
IKLAN           1
Sudan           1
Rafah           1
Beirut          1
ALSHAM          1
ESSAOUIRA       1
Pointe          1
Seef            1
Force           1
Camp            1
Men-O-Lan       1
Arak            1
Name: count, dtype: int64